# ISTAT SDMX API – Catalogo Dataflow

Questo notebook recupera tutti i **dataflow disponibili** dall'API SDMX di ISTAT e li carica in un DataFrame pandas per esplorazione e analisi.

**Riferimento:** [Guida API ISTAT – onData](https://ondata.github.io/guida-api-istat/)

**Endpoint base:** `https://esploradati.istat.it/SDMXWS/rest/dataflow/IT1`

> ⚠️ **Attenzione:** L'API ISTAT ha un limite di **5 richieste al minuto** per IP. Superarlo comporta un blocco di 1-2 giorni.

In [ ]:
# Le librerie sono già importate nella cella fetch – questa cella è opzionale
print("Pronto.")

In [ ]:
import pandas as pd
import requests
import re
import warnings
warnings.filterwarnings('ignore')

BASE_URL = "https://esploradati.istat.it/SDMXWS/rest"
HEADERS  = {"Accept": "application/vnd.sdmx.data+csv;version=1.0.0"}

print("Scarico lista dataflow...")
r = requests.get(f"{BASE_URL}/dataflow/IT1", timeout=120)
print(f"Status: {r.status_code} – {len(r.content) / 1024:.1f} KB ricevuti")

In [ ]:
# Estrai id e nome italiano con regex (stesso approccio del codice funzionante)
matches = re.findall(
    r'<structure:Dataflow id="([^"]+)".*?<common:Name xml:lang="it">([^<]+)</common:Name>',
    r.text, re.DOTALL
)

# Estrai anche il nome inglese
matches_en = re.findall(
    r'<structure:Dataflow id="([^"]+)".*?<common:Name xml:lang="en">([^<]+)</common:Name>',
    r.text, re.DOTALL
)
en_dict = dict(matches_en)

# Estrai iddatastructure
matches_dsd = re.findall(
    r'<structure:Dataflow id="([^"]+)".*?<Ref id="([^"]+)"',
    r.text, re.DOTALL
)
dsd_dict = dict(matches_dsd)

print(f"Dataflow trovati: {len(matches)}")

In [ ]:
records = []
for id_dataflow, descrizione in matches:
    id_dsd  = dsd_dict.get(id_dataflow, '')
    records.append({
        'iddataflow':      id_dataflow,
        'descrizione':     descrizione,
        'descrizione_en':  en_dict.get(id_dataflow, ''),
        'iddatastructure': id_dsd,
        'url':             f'http://dati.istat.it/Index.aspx?DataSetCode={id_dsd}' if id_dsd else '',
    })

df_dataflows = pd.DataFrame(records).sort_values('descrizione').reset_index(drop=True)

print(f"DataFrame: {df_dataflows.shape[0]} righe × {df_dataflows.shape[1]} colonne")
df_dataflows.head(10)

In [ ]:
# -------------------------------------------------------------------
# Esplorazione del DataFrame
# -------------------------------------------------------------------

# Ricerca per parola chiave (modifica il termine qui sotto)
KEYWORD = 'popolazione'

mask = (
    df_dataflows['descrizione'].str.contains(KEYWORD, case=False, na=False) |
    df_dataflows['descrizione_en'].str.contains(KEYWORD, case=False, na=False)
)
risultati = df_dataflows[mask]

print(f'Risultati per "{KEYWORD}": {len(risultati)}')
risultati[['iddataflow', 'descrizione', 'descrizione_en', 'url']]

In [ ]:
# -------------------------------------------------------------------
# (Opzionale) Salva il catalogo in CSV
# -------------------------------------------------------------------
OUTPUT_FILE = 'istat_dataflows.csv'
df_dataflows.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f'Catalogo salvato in: {OUTPUT_FILE}')